# <center> 355. Design Twitter </center>


## Problem Description
[Click here](https://leetcode.com/problems/design-twitter/description/)


## Intuition
<!-- Describe your first thoughts on how to solve this problem. -->
For posts/tweets, use a hashmap with key as the user id and value as a list (stack) to store tweets because we need the latest tweet at the top. A stack allow insertion and deletion at the top in O(1) time. 

For the following list, use a hashmap with key as follower id and value as a set of followee ids to insert (follow user) and delete (unfollow user) in O(1) time.

To get the news feed, we need the 10 most recent tweets posted by the user and their followees. Add a timestamp to every tweet and maintain a timer in decreasing order so that newer tweets have smaller timestamps than older tweets.

To get the most recent tweets (tweets with the smallest timestamps), use a min heap because it returns the smallest element in O(1) time. Push the latest tweet from each user (user + followees) into the heap. Each time pop the most recent tweet and push the next older tweet from the same user.


## Approach
<!-- Describe your approach to solving the problem. -->
**init()**
- set timer = 0 to track the timestamp
- create a hashmap for tweets
    - *key = user id*
    - *value = list of (timestamp, tweetId) stored in increasing order of time*
- create a hashmap for followees
    - *key = follower id*
    - *value = set of followee ids*

**postTweet()**
- decrement the timer so that newer tweets have smaller timestamps than older tweets
- add the tweet (timer, tweet id) to the tweets stack of the user

**follow()**
- add the followee id to the set of followees for the follower

**unfollow()**
- remove the followee id from the set of followees for the follower, if the followee exists
    - *set discard() will handle the check for the followee because it removes the item if exists, and does nothing if the item doesn't exist, unlike set remove()*

**getNewsFeed()**
- initialize a min heap
- set news_feed = a list to store the top 10 tweets
- get all relevant followed users i.e user + followees
- for each user in followed_users
  - take their latest tweet (last element in list)
  - push into heap as (-time, tweetId, user, index_of_previous_tweet)
- loop while heap is not empty and we have less than 10 tweets
  - pop the most recent tweet
  - add tweetId to result
  - if the same user has older tweets:
    - push the next older tweet into heap
- return the news_feed (top 10 most recent tweets)


## Complexity
<!-- Add your time complexity here, e.g. $$O(n)$$ -->
- Time complexity:
    - init(): O(1)
    - postTweet(): O(1)
    - follow(): O(1)
    - unfollow(): O(1)
    - getNewsFeed(): O(building heap + extracting top 10 tweets) → O(users * heap push + 10 * heap push/pop) → O(u*logu + 10*logu) → O(ulogu)
      - *we build a heap from the latest tweet of each followed user*

<!-- Add your space complexity here, e.g. $$O(n)$$ -->
- Space complexity:
    - init(): O(1)
    - postTweet(): O(1)
    - follow(): O(1)
    - unfollow(): O(1)
    - getNewsFeed(): O(heap + news feed list) → O(users + top 10 tweets) → O(u + 10) → O(u)
      - *heap stores at most one entry per followed user*
    - overall space: O(tweets hashmap + followees hashmap) → O(t + f)


## Code

In [ ]:
class Twitter:

    def __init__(self):
        self.timer = 0
        self.tweets = collections.defaultdict(list)
        self.followees = collections.defaultdict(set)

    def postTweet(self, userId: int, tweetId: int) -> None:
        self.timer -= 1
        self.tweets[userId].append((self.timer, tweetId))

    def follow(self, followerId: int, followeeId: int) -> None:
        self.followees[followerId].add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        self.followees[followerId].discard(followeeId)

    def getNewsFeed(self, userId: int) -> List[int]:
        min_heap = []
        news_feed = []
        followed_users = self.followees[userId] | {userId}
        for user in followed_users:
            if self.tweets[user]:
                tweet_index = len(self.tweets[user]) - 1
                time, tweet_id = self.tweets[user][tweet_index]
                heapq.heappush(min_heap, (-time, tweet_id, user, tweet_index - 1))
        while min_heap and len(news_feed) < 10:
            time, tweet_id, user, tweet_index = heapq.heappop(min_heap)
            news_feed.append(tweet_id)
            if tweet_index >= 0:
                next_time, next_tweet_id = self.tweets[user][tweet_index]
                heapq.heappush(min_heap, (-next_time, next_tweet_id, user, tweet_index - 1))
        return news_feed
        


# Your Twitter object will be instantiated and called as such:
# obj = Twitter()
# obj.postTweet(userId,tweetId)
# param_2 = obj.getNewsFeed(userId)
# obj.follow(followerId,followeeId)
# obj.unfollow(followerId,followeeId)